In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv
import os
from datetime import datetime, timezone

load_dotenv(find_dotenv())
DB_URL = os.getenv("LOCAL_DATABASE_URL")

if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

if not DB_URL:
    raise ValueError("LOCAL_DATABASE_URL tidak ditemukan di .env")

engine = create_engine(DB_URL)

print("Database connection berhasil")
print(f"Target: {DB_URL.split('@')[1] if '@' in DB_URL else 'configured database'}")


Database connection berhasil
Target: localhost:5432/retail_analytics


In [3]:
order_360_test = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY updated_at_utc DESC, ingested_at_utc DESC, source_row_id
                   ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ),

    order_items_summary AS (
        SELECT
            order_id,
            COUNT(order_item_id) AS item_count,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_merchandise_value,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    ),

    payment_events_dedup AS (
        SELECT DISTINCT
            event_id,
            "payload.order_id" AS order_id,
            "payload.amount" AS payment_amount
        FROM silver.payment_events
        WHERE event_type = 'PAYMENT_CAPTURED'
    ),

    payment_summary AS (
        SELECT
            order_id,
            SUM(payment_amount) AS captured_payment_amount
        FROM payment_events_dedup
        GROUP BY order_id
    ),

    payment_status_summary AS (
        SELECT order_id, event_type AS payment_status
        FROM (
            SELECT
                "payload.order_id" AS order_id,
                event_type,
                occurred_at_utc,
                event_id,
                ROW_NUMBER() OVER (
                    PARTITION BY "payload.order_id"
                    ORDER BY occurred_at_utc DESC, event_id DESC
                ) AS rn
            FROM silver.payment_events
        ) x
        WHERE rn = 1
    ),

    refund_dedup AS (
        SELECT
            "payload.order_id" AS order_id,
            "payload.refund_id" AS refund_id,
            MAX("payload.amount") AS refund_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY "payload.order_id", "payload.refund_id"
    ),

    refund_summary AS (
        SELECT order_id, SUM(refund_amount) AS refunded_amount
        FROM refund_dedup
        GROUP BY order_id
    ),

    return_summary AS (
        SELECT order_id, event_type AS return_status
        FROM (
            SELECT
                "payload.order_id" AS order_id,
                event_type,
                occurred_at_utc,
                event_id,
                ROW_NUMBER() OVER (
                    PARTITION BY "payload.order_id"
                    ORDER BY occurred_at_utc DESC, event_id DESC
                ) AS rn
            FROM silver.return_events
        ) x
        WHERE rn = 1
    ),

    promotion_summary AS (
        SELECT
            order_id,
            COUNT(DISTINCT promotion_id) AS promotion_count
        FROM silver.order_promotions
        GROUP BY order_id
    ),

    first_order_summary AS (
        SELECT
            order_id,
            (ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) = 1) AS first_order_flag
        FROM orders_dedup
    )

    SELECT
        o.order_id,
        o.customer_id,
        o.ordered_at_utc::DATE AS order_date,
        o.sales_channel,
        o.store_id,
        COALESCE(oi.gross_merchandise_value, 0) AS gross_merchandise_value,
        COALESCE(oi.discount_amount, 0) AS discount_amount,
        COALESCE(o.shipping_revenue, 0) AS shipping_revenue,
        COALESCE(ps.captured_payment_amount, 0) AS captured_payment_amount,
        COALESCE(rf.refunded_amount, 0) AS refunded_amount,
        (
            COALESCE(oi.gross_merchandise_value, 0)
            - COALESCE(oi.discount_amount, 0)
            + COALESCE(o.shipping_revenue, 0)
            - COALESCE(rf.refunded_amount, 0)
        ) AS net_revenue,
        COALESCE(oi.item_count, 0) AS item_count,
        COALESCE(oi.unit_quantity, 0) AS unit_quantity,
        o.status AS order_status,
        COALESCE(pst.payment_status, 'NO_PAYMENT') AS payment_status,
        COALESCE(rs.return_status, 'NO_RETURN') AS return_status,
        COALESCE(pr.promotion_count, 0) AS promotion_count,
        fo.first_order_flag
    FROM orders_dedup o
    LEFT JOIN order_items_summary oi ON o.order_id = oi.order_id
    LEFT JOIN payment_summary ps ON o.order_id = ps.order_id
    LEFT JOIN payment_status_summary pst ON o.order_id = pst.order_id
    LEFT JOIN refund_summary rf ON o.order_id = rf.order_id
    LEFT JOIN return_summary rs ON o.order_id = rs.order_id
    LEFT JOIN promotion_summary pr ON o.order_id = pr.order_id
    LEFT JOIN first_order_summary fo ON o.order_id = fo.order_id
    ORDER BY o.order_id
    """,
    engine
)

print("Rows:", len(order_360_test))
print("Unique order_id:", order_360_test["order_id"].nunique())
display(order_360_test.head(10))


Rows: 10000
Unique order_id: 10000


,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,item_count,unit_quantity,order_status,payment_status,return_status,promotion_count,first_order_flag
0,ORD-000001,CUST-01757,2026-08-20,MOBILE_APP,None,1118.24,0.0,12.0,1125.24,0.0,1130.24,2,4.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,1,False
1,ORD-000002,CUST-01160,2026-08-16,MOBILE_APP,None,1503.35,0.0,0.0,1490.35,0.0,1503.35,2,5.0,RETURNED,PAYMENT_CAPTURED,NO_RETURN,2,False
2,ORD-000003,CUST-00962,2026-07-04,WEB,None,278.49,5.0,7.5,270.99,0.0,280.99,1,3.0,PLACED,PAYMENT_AUTHORIZED,RETURN_REQUESTED,1,True
3,ORD-000004,CUST-00181,2026-08-28,MARKETPLACE,None,2396.58,5.0,0.0,2361.58,0.0,2391.58,3,11.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
4,ORD-000005,CUST-02298,2026-07-06,STORE,STORE-03,1883.20,5.0,7.5,1867.70,0.0,1885.70,4,8.0,RETURNED,PAYMENT_AUTHORIZED,NO_RETURN,2,False
5,ORD-000006,CUST-00630,2026-08-08,WEB,None,2200.22,0.0,0.0,0.00,0.0,2200.22,4,11.0,CANCELLED,PAYMENT_FAILED,NO_RETURN,1,False
6,ORD-000007,CUST-00632,2026-07-18,STORE,STORE-03,2131.88,10.0,0.0,2482.04,0.0,2121.88,4,7.0,FULFILLED,PAYMENT_CAPTURED,NO_RETURN,2,False
7,ORD-000008,CUST-01792,2026-08-28,WEB,None,980.93,5.0,0.0,955.93,0.0,975.93,3,7.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
8,ORD-000009,CUST-01220,2026-07-15,WEB,None,2812.32,0.0,3.5,2800.82,0.0,2815.82,4,9.0,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN,1,False
9,ORD-000010,CUST-00459,2026-07-30,MOBILE_APP,None,1503.56,0.0,0.0,1473.56,0.0,1503.56,1,4.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False


In [4]:
expected_columns = [
    "order_id", "customer_id", "order_date", "sales_channel", "store_id",
    "gross_merchandise_value", "discount_amount", "shipping_revenue",
    "captured_payment_amount", "refunded_amount", "net_revenue",
    "item_count", "unit_quantity", "order_status", "payment_status",
    "return_status", "promotion_count", "first_order_flag"
]

print("Jumlah kolom aktual:", len(order_360_test.columns))
print("Jumlah kolom expected:", len(expected_columns))
print("Kolom belum ada:", [c for c in expected_columns if c not in order_360_test.columns])
print("Kolom tambahan:", [c for c in order_360_test.columns if c not in expected_columns])


Jumlah kolom aktual: 18
Jumlah kolom expected: 18
Kolom belum ada: []
Kolom tambahan: []


In [5]:
null_check = order_360_test[expected_columns].isna().sum()
print("NULL pada kolom:")
display(null_check[null_check > 0])

print("Total NULL:", int(null_check.sum()))


NULL pada kolom:


store_id    7483
dtype: int64

Total NULL: 7483


In [6]:
duplicate_rows = len(order_360_test) - order_360_test["order_id"].nunique()
print("Total rows:", len(order_360_test))
print("Unique order_id:", order_360_test["order_id"].nunique())
print("Duplicate rows:", duplicate_rows)


Total rows: 10000
Unique order_id: 10000
Duplicate rows: 0


In [7]:
net_revenue_check = order_360_test[
    order_360_test["net_revenue"].round(2) != (
        order_360_test["gross_merchandise_value"]
        - order_360_test["discount_amount"]
        + order_360_test["shipping_revenue"]
        - order_360_test["refunded_amount"]
    ).round(2)
]
print("Net revenue mismatch:", len(net_revenue_check))


Net revenue mismatch: 0


In [8]:
payment_anomaly = order_360_test[
    (order_360_test["captured_payment_amount"] < 0)
    | ((order_360_test["payment_status"] == "PAYMENT_CAPTURED") &
       (order_360_test["captured_payment_amount"] <= 0))
]
print("Payment anomaly (captured status but amount <= 0):", len(payment_anomaly))
if len(payment_anomaly):
    display(payment_anomaly[["order_id", "payment_status", "captured_payment_amount"]].head(20))


Payment anomaly (captured status but amount <= 0): 8


,order_id,payment_status,captured_payment_amount
510,ORD-000511,PAYMENT_CAPTURED,0.0
2278,ORD-002279,PAYMENT_CAPTURED,0.0
2405,ORD-002406,PAYMENT_CAPTURED,0.0
3651,ORD-003652,PAYMENT_CAPTURED,0.0
5052,ORD-005053,PAYMENT_CAPTURED,0.0
7665,ORD-007666,PAYMENT_CAPTURED,0.0
7705,ORD-007706,PAYMENT_CAPTURED,0.0
8158,ORD-008159,PAYMENT_CAPTURED,0.0


In [9]:
refund_check = order_360_test[order_360_test["refunded_amount"] < 0]
print("Refund amount negatif:", len(refund_check))


Refund amount negatif: 0


In [10]:
silver_refund_total = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT
            "payload.order_id" AS order_id,
            "payload.refund_id" AS refund_id,
            MAX("payload.amount") AS refund_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY "payload.order_id", "payload.refund_id"
    )
    SELECT SUM(refund_amount) AS total_refund FROM refund_dedup
    """, engine
).iloc[0]["total_refund"]

order_360_refund_total = order_360_test["refunded_amount"].sum()
print("Silver refund total:", round(float(silver_refund_total), 2))
print("Order 360 refund total:", round(float(order_360_refund_total), 2))
print("Selisih:", round(float(silver_refund_total) - float(order_360_refund_total), 2))


Silver refund total: 1549240.21
Order 360 refund total: 1549240.21
Selisih: 0.0


In [11]:
return_compare = pd.read_sql(
    """
    WITH ranked_returns AS (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.return_events
    )
    SELECT order_id, event_type AS silver_return_status
    FROM ranked_returns
    WHERE rn = 1
    """, engine
).merge(order_360_test[["order_id", "return_status"]], on="order_id", how="left")

return_compare["return_status"] = return_compare["return_status"].fillna("NO_RETURN")
return_mismatch = return_compare[return_compare["silver_return_status"] != return_compare["return_status"]]
print("Return status mismatch:", len(return_mismatch))


Return status mismatch: 0


In [12]:
first_order_compare = pd.read_sql(
    """
    WITH dedup_orders AS (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY updated_at_utc DESC, ingested_at_utc DESC, source_row_id
            ) AS rn
            FROM silver.orders
        ) x WHERE rn = 1
    ), ranked_orders AS (
        SELECT order_id, customer_id,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ordered_at_utc, order_id) AS customer_order_rank
        FROM dedup_orders
    )
    SELECT order_id, (customer_order_rank = 1) AS expected_first_order_flag
    FROM ranked_orders
    """, engine
).merge(order_360_test[["order_id", "first_order_flag"]], on="order_id", how="left")

first_order_mismatch = first_order_compare[
    first_order_compare["expected_first_order_flag"] != first_order_compare["first_order_flag"]
]
print("First order flag mismatch:", len(first_order_mismatch))


First order flag mismatch: 0


In [13]:
print("=== FINAL ORDER 360 CHECK ===")
print("Rows:", len(order_360_test))
print("Unique orders:", order_360_test["order_id"].nunique())
print("Duplicate rows:", len(order_360_test) - order_360_test["order_id"].nunique())
print("Net revenue mismatch:", len(net_revenue_check))
print("Payment anomaly:", len(payment_anomaly))
print("Negative refund:", len(refund_check))
print("Return status mismatch:", len(return_mismatch))
print("First order mismatch:", len(first_order_mismatch))


=== FINAL ORDER 360 CHECK ===
Rows: 10000
Unique orders: 10000
Duplicate rows: 0
Net revenue mismatch: 0
Payment anomaly: 8
Negative refund: 0
Return status mismatch: 0
First order mismatch: 0


In [14]:
pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
order_360_gold = order_360_test.copy()
order_360_gold["pipeline_run_id"] = pipeline_run_id

# Final numeric type preparation
order_360_gold["item_count"] = order_360_gold["item_count"].astype(int)
order_360_gold["unit_quantity"] = order_360_gold["unit_quantity"].astype(int)
order_360_gold["promotion_count"] = order_360_gold["promotion_count"].astype(int)

print("Pipeline run ID:", pipeline_run_id)
print("Rows:", len(order_360_gold))
print("Columns:", len(order_360_gold.columns))


Pipeline run ID: 20260924070551
Rows: 10000
Columns: 19


In [15]:
expected_gold_columns = expected_columns + ["pipeline_run_id"]
print("Jumlah kolom aktual:", len(order_360_gold.columns))
print("Jumlah kolom expected:", len(expected_gold_columns))
print("Kolom belum ada:", [c for c in expected_gold_columns if c not in order_360_gold.columns])
print("Kolom tambahan:", [c for c in order_360_gold.columns if c not in expected_gold_columns])


Jumlah kolom aktual: 19
Jumlah kolom expected: 19
Kolom belum ada: []
Kolom tambahan: []


In [16]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS gold"))
    conn.execute(text("DROP TABLE IF EXISTS gold.order_360"))
    conn.execute(text("""
        CREATE TABLE gold.order_360 (
            order_id TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL,
            order_date DATE NOT NULL,
            sales_channel TEXT NOT NULL,
            store_id TEXT,
            gross_merchandise_value NUMERIC(14, 2) NOT NULL,
            discount_amount NUMERIC(14, 2) NOT NULL,
            shipping_revenue NUMERIC(14, 2) NOT NULL,
            captured_payment_amount NUMERIC(14, 2) NOT NULL,
            refunded_amount NUMERIC(14, 2) NOT NULL,
            net_revenue NUMERIC(14, 2) NOT NULL,
            item_count INTEGER NOT NULL,
            unit_quantity INTEGER NOT NULL,
            order_status TEXT NOT NULL,
            payment_status TEXT NOT NULL,
            return_status TEXT NOT NULL,
            promotion_count INTEGER NOT NULL,
            first_order_flag BOOLEAN NOT NULL,
            pipeline_run_id TEXT NOT NULL
        )
    """))

order_360_gold.to_sql(
    "order_360",
    engine,
    schema="gold",
    if_exists="append",
    index=False,
    method="multi"
)

print("gold.order_360 berhasil dibuat dan di-load.")
print("Rows inserted:", len(order_360_gold))


gold.order_360 berhasil dibuat dan di-load.
Rows inserted: 10000


In [17]:
gold_order_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_rows
    FROM gold.order_360
    """, engine
)
display(gold_order_check)


,total_rows,unique_orders,duplicate_rows
0,10000,10000,0


In [18]:
gold_schema_check = pd.read_sql(
    """
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'gold'
      AND table_name = 'order_360'
    ORDER BY ordinal_position
    """, engine
)
display(gold_schema_check)


,column_name,data_type,is_nullable
0,order_id,text,NO
1,customer_id,text,NO
2,order_date,date,NO
3,sales_channel,text,NO
4,store_id,text,YES
5,gross_merchandise_value,numeric,NO
6,discount_amount,numeric,NO
7,shipping_revenue,numeric,NO
8,captured_payment_amount,numeric,NO
9,refunded_amount,numeric,NO


In [19]:
gold_null_check = pd.read_sql(
    """
    SELECT
        COUNT(*) FILTER (WHERE order_id IS NULL) AS order_id,
        COUNT(*) FILTER (WHERE customer_id IS NULL) AS customer_id,
        COUNT(*) FILTER (WHERE order_date IS NULL) AS order_date,
        COUNT(*) FILTER (WHERE sales_channel IS NULL) AS sales_channel,
        COUNT(*) FILTER (WHERE gross_merchandise_value IS NULL) AS gross_merchandise_value,
        COUNT(*) FILTER (WHERE discount_amount IS NULL) AS discount_amount,
        COUNT(*) FILTER (WHERE shipping_revenue IS NULL) AS shipping_revenue,
        COUNT(*) FILTER (WHERE captured_payment_amount IS NULL) AS captured_payment_amount,
        COUNT(*) FILTER (WHERE refunded_amount IS NULL) AS refunded_amount,
        COUNT(*) FILTER (WHERE net_revenue IS NULL) AS net_revenue,
        COUNT(*) FILTER (WHERE item_count IS NULL) AS item_count,
        COUNT(*) FILTER (WHERE unit_quantity IS NULL) AS unit_quantity,
        COUNT(*) FILTER (WHERE order_status IS NULL) AS order_status,
        COUNT(*) FILTER (WHERE payment_status IS NULL) AS payment_status,
        COUNT(*) FILTER (WHERE return_status IS NULL) AS return_status,
        COUNT(*) FILTER (WHERE promotion_count IS NULL) AS promotion_count,
        COUNT(*) FILTER (WHERE first_order_flag IS NULL) AS first_order_flag,
        COUNT(*) FILTER (WHERE pipeline_run_id IS NULL) AS pipeline_run_id
    FROM gold.order_360
    """, engine
)

display(gold_null_check.T)


,0
order_id,0
customer_id,0
order_date,0
sales_channel,0
gross_merchandise_value,0
discount_amount,0
shipping_revenue,0
captured_payment_amount,0
refunded_amount,0
net_revenue,0


In [20]:
gold_net_revenue_check = pd.read_sql(
    """
    SELECT COUNT(*) AS mismatch_count
    FROM gold.order_360
    WHERE ROUND(net_revenue, 2) != ROUND(
        gross_merchandise_value - discount_amount + shipping_revenue - refunded_amount,
        2
    )
    """, engine
)
display(gold_net_revenue_check)


,mismatch_count
0,0


In [21]:
gold_payment_check = pd.read_sql(
    """
    SELECT
        payment_status,
        COUNT(*) AS total_orders,
        COUNT(*) FILTER (WHERE captured_payment_amount > 0) AS orders_with_payment,
        COUNT(*) FILTER (WHERE captured_payment_amount = 0) AS orders_without_payment
    FROM gold.order_360
    GROUP BY payment_status
    ORDER BY payment_status
    """, engine
)
display(gold_payment_check)


,payment_status,total_orders,orders_with_payment,orders_without_payment
0,PAYMENT_AUTHORIZED,881,771,110
1,PAYMENT_CAPTURED,8013,8005,8
2,PAYMENT_FAILED,1106,0,1106


In [22]:
gold_payment_reconciliation = pd.read_sql(
    """
    WITH deduplicated_payment_events AS (
        SELECT DISTINCT
            event_id,
            "payload.order_id" AS order_id,
            "payload.amount" AS payment_amount
        FROM silver.payment_events
        WHERE event_type = 'PAYMENT_CAPTURED'
    ),
    silver_payment AS (
        SELECT
            order_id,
            ROUND(SUM(payment_amount)::numeric, 2) AS silver_captured_payment
        FROM deduplicated_payment_events
        GROUP BY order_id
    )
    SELECT
        g.order_id,
        g.captured_payment_amount,
        COALESCE(s.silver_captured_payment, 0) AS silver_captured_payment,
        ROUND((g.captured_payment_amount - COALESCE(s.silver_captured_payment, 0))::numeric, 2) AS difference
    FROM gold.order_360 g
    LEFT JOIN silver_payment s ON g.order_id = s.order_id
    WHERE ROUND((g.captured_payment_amount - COALESCE(s.silver_captured_payment, 0))::numeric, 2) != 0
    ORDER BY ABS(g.captured_payment_amount - COALESCE(s.silver_captured_payment, 0)) DESC
    """, engine
)

print("Payment reconciliation mismatch:", len(gold_payment_reconciliation))
display(gold_payment_reconciliation.head(20))


Payment reconciliation mismatch: 0


,order_id,captured_payment_amount,silver_captured_payment,difference


In [23]:
gold_first_order_check = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY updated_at_utc DESC, ingested_at_utc DESC, source_row_id
                   ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ),
    customer_order_rank AS (
        SELECT
            order_id,
            customer_id,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) AS customer_order_rank
        FROM orders_dedup
    )
    SELECT COUNT(*) FILTER (
        WHERE g.first_order_flag != (c.customer_order_rank = 1)
    ) AS first_order_mismatch
    FROM gold.order_360 g
    JOIN customer_order_rank c ON g.order_id = c.order_id
    """, engine
)
display(gold_first_order_check)


,first_order_mismatch
0,0


In [24]:
print("=== GOLD.ORDER_360 FINAL SUMMARY ===")
print("Pipeline run ID:", pipeline_run_id)
print("Rows:", len(order_360_gold))
print("Unique orders:", order_360_gold["order_id"].nunique())
print("Gold duplicate rows:", int(gold_order_check.iloc[0]["duplicate_rows"]))
print("Gold NULL total:", int(gold_null_check.sum(axis=1).iloc[0]))
print("Net revenue mismatch:", int(gold_net_revenue_check.iloc[0]["mismatch_count"]))
print("Payment reconciliation mismatch:", len(gold_payment_reconciliation))
print("First order mismatch:", int(gold_first_order_check.iloc[0]["first_order_mismatch"]))
print()
print("Gold order_360 selesai.")


=== GOLD.ORDER_360 FINAL SUMMARY ===
Pipeline run ID: 20260924070551
Rows: 10000
Unique orders: 10000
Gold duplicate rows: 0
Gold NULL total: 0
Net revenue mismatch: 0
Payment reconciliation mismatch: 0
First order mismatch: 0

Gold order_360 selesai.


In [25]:
schema_check = pd.read_sql("""
SELECT
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'gold'
  AND table_name = 'order_360'
ORDER BY ordinal_position
""", engine)

display(schema_check)

,ordinal_position,column_name,data_type,is_nullable
0,1,order_id,text,NO
1,2,customer_id,text,NO
2,3,order_date,date,NO
3,4,sales_channel,text,NO
4,5,store_id,text,YES
5,6,gross_merchandise_value,numeric,NO
6,7,discount_amount,numeric,NO
7,8,shipping_revenue,numeric,NO
8,9,captured_payment_amount,numeric,NO
9,10,refunded_amount,numeric,NO
